# Pipelines:

The HuggingFace transformers library provides APIs at two different levels.

The High Level API for using open-source models for typical inference tasks is called "pipelines". It's incredibly easy to use.

You create a pipeline using something like:
`my_pipeline = pipeline("the_task_I_want_to_do")`

Followed by
`result = my_pipeline(my_input)`

And that's it!

## A sidenote:

When working with Data Science models, you could be carrying out 2 very different activities: **training** and **inference**.

### 1. Training

**Training** is when you provide a model with data for it to adapt to get better at a task in the future. It does this by updating its internal settings - the parameters or weights of the model. If you're Training a model that's already had some training, the activity is called "fine-tuning".

### 2. Inference

**Inference** is when you are working with a model that has _already been trained_. You are using that model to produce new outputs on new inputs, taking advantage of everything it learned while it was being trained. Inference is also sometimes referred to as "Execution" or "Running a model".

All of our use of APIs for GPT, Claude and Gemini in the last weeks are examples of **inference**. The "P" in GPT stands for "Pre-trained", meaning that it has already been trained with data (lots of it!).

The pipelines API in HuggingFace is only for use for **inference** - running a model that has already been trained.

See the below video for more on parameters, training and inference:
https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs


In [4]:
# Checking GPU Access:

import torch

gpu_availability = torch.cuda.is_available()
print(f'GPU Availability: {gpu_availability}')
print(f'GPU Count: {torch.cuda.device_count()}')
print(f'GPU Name: {torch.cuda.get_device_name(torch.cuda.current_device())}')

GPU Availability: True
GPU Count: 1
GPU Name: NVIDIA GeForce RTX 4080 Laptop GPU


In [5]:
# Connecting to Hugging Face Account:

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

hf_token = os.getenv('HF_TOKEN')

if hf_token and hf_token.startswith('hf_'):
    print('HF Key found!')
    login(token= hf_token, add_to_git_credential= True)
else:
    print('HF Key not found!')

HF Key found!


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Using Pipelines from Hugging Face

A simple way to run inference for common tasks, without worrying about all the plumbing, picking reasonable defaults.


### How it works:

STEP 1: Create a pipeline - a function you can then call

```python
my_pipeline = pipeline(task, model=xx, device=xx)
```

If you don't specify a model, then Hugging Face picks one for you that's the default for the task. Specify "cuda" for the device to use an NVIDIA GPU like the one on the T4. Specify "mps" on a Mac.


STEP 2: Then call it as many times as you want:

```python
my_pipeline(input1)
my_pipeline(input2)
```

In [8]:
# Imports:

import torch
from transformers import pipeline
from datasets import load_dataset

import warnings
warnings.filterwarnings("ignore")

**Each Cell below will download a different Hugging Face model depending on the task (whether specified or not), and will take some time to run therefore.**

### Sentiment Analysis:

In [9]:
# Without Specifying Model:

simple_sentiment_analyser = pipeline(task= 'sentiment-analysis', device= 'cuda')

result = simple_sentiment_analyser('I am super excited to be on the way to LLM Mastery!')

print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.999354898929596}]


In [11]:
result = simple_sentiment_analyser('I should be more excited to be on the way to LLM mastery!')

print(result)

[{'label': 'POSITIVE', 'score': 0.9008278250694275}]


In [12]:
# Specifying Model:

better_sentiment_analyser = pipeline(task= 'sentiment-analysis', device= 'cuda', model="nlptown/bert-base-multilingual-uncased-sentiment")

result = better_sentiment_analyser('I should be more excited to be on the way to LLM mastery!')

print(result)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': '3 stars', 'score': 0.3901217579841614}]


In [13]:
result = better_sentiment_analyser('I am super excited to be on the way to LLM Mastery!')

print(result)

[{'label': '5 stars', 'score': 0.7223415970802307}]


### Named Entity Recognition:

In [15]:
ner = pipeline(task= 'ner', device= 'cuda')
result = ner('AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner')

for entity in result:
    print(f"{entity['word']} : {entity['entity']}")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AI : I-ORG
Engineers : I-ORG
Hu : I-ORG
##gging : I-ORG
##F : I-ORG
##ace : I-ORG
Google : I-MISC
Cola : I-MISC
Ed : I-PER
Don : I-PER
##ner : I-PER
